# Lab 09 · Evaluation: a multi-agent system against a single model

**Prepared by Mustafa Assaf** · Self-study: Go deeper · 15–20 minute lab

## Evaluate recommendations against lecture scenarios

**New:** compare single-model and council answers on the *same* labelled cases. The examples revisit slide 30: broad support search suggests CQRS; a small order-details lookup by ID suggests API composition.

The table below contains **invented teaching fixtures**, including call counts and timings. They were not measured from Qwen, DeepSeek, Gemini, or a council, and must not be reported as model benchmarks. Switching to live mode runs only the judge. For a real experiment, collect fresh answers, actual wall-clock time, token usage, and human ratings from every system.

We distinguish label agreement, valid citation IDs, and a stated downside. A valid ID does not prove that a source supports the claim.

**At a glance**

![Lab 09 at a glance: Synthetic run fixtures (4 systems × 2 cases), then Correct · citation IDs (calls · seconds), then LLM judge (names a cost?), then Calibrate (agree with human ≥ 75%), then Council vs best (wins, and at what cost).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/09.svg)

## Review your microservices lectures

- **Microservices & Their Patterns**, slides 25, 27, 28, 29, 30.

**Recall:** Choose data patterns from requirements; Measure label agreement, citation validity, and trade-offs.

Before running the example:

1. Why is CQRS excessive for a small lookup by ID?
2. Which data pattern supports replaying a historical account balance?

Slide numbers count from the title slide, including appendices. The small scenarios and code are teaching adaptations, not production services. Use the lecture decks supplied in class; they are not uploaded with this course.

## The idea in context

Evaluation needs consistent inputs, labelled criteria, and honest provenance. Synthetic answers and timings teach the method without claiming measured model performance.

Use actual run logs and repeated scenarios for the project experiment.

| Key term | Meaning |
| --- | --- |
| **Baseline** | The system to beat; here, the best single model. |
| **Metric** | A number that measures one quality, such as correct answers or model calls. |
| **LLM judge** | A model that rates other models' answers against a rubric. |
| **Calibration** | Checking the judge against human ratings before trusting it. |

## Pattern · Evaluating a pattern

This lab is not a pattern of its own. It measures the quality attributes that the pattern cards in Labs 06–08 name, such as Accuracy, Cost, and Latency, for *Voting-Based Cooperation* and *Debate-Based Cooperation*. The LLM judge is one of the aggregation options that the field guide lists for Voting-Based Cooperation; here it rates answers, and it is trusted only after calibration.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Two scenarios and synthetic answers

The expected labels are introductory choices from slides 27–30, not unique answers to every real architecture problem. Each tuple holds the option, citations, explanation, illustrative calls, and illustrative seconds.

In [ ]:
import json

SOURCES = {"M27", "M28", "M30"}
expert = {"Support search": "CQRS", "Order details": "API composition"}
runs = {  # SYNTHETIC fixtures, not observed provider outputs or measured timings
    ("Qwen", "Support search"): ("API composition", ["M27"], "One composer is simple.", 1, 1.6),
    ("Qwen", "Order details"): ("API composition", ["M27"], "Lookup by ID fits a composer.", 1, 1.5),
    ("DeepSeek", "Support search"): ("CQRS", ["M28"], "Rich search is fast, but the view may lag.", 1, 2.2),
    ("DeepSeek", "Order details"): ("API composition", [], "A composer is easy.", 1, 2.0),
    ("Gemini", "Support search"): ("API composition", [], "A composer is always best.", 1, 1.9),
    ("Gemini", "Order details"): ("API composition", ["M27"], "Small lookup; a source outage can fail it.", 1, 2.1),
    ("Council", "Support search"): ("CQRS", ["M28", "M30"], "Rich search, at the cost of a stale extra view.", 5, 9.8),
    ("Council", "Order details"): ("API composition", ["M27"], "Small lookup; all source services must respond.", 5, 9.1),
}
systems = ["Qwen", "DeepSeek", "Gemini", "Council"]

## Step 2 · Compute explicit measures

Every system must answer both cases. `citation_valid` checks known IDs only. The seconds and calls merely demonstrate how accounting works; replace them with actual measurements in a project.

In [ ]:
assert all((s, sc) in runs for s in systems for sc in expert), "Every system must answer every scenario."

def measure(system):
    rows = [(scenario, run) for (s, scenario), run in runs.items() if s == system]
    return {"correct": sum(run[0] == expert[scenario] for scenario, run in rows),
            "citation_valid": sum(bool(run[1]) and set(run[1]) <= SOURCES for _, run in rows),
            "calls": sum(run[3] for _, run in rows), "seconds": round(sum(run[4] for _, run in rows), 1)}

scores = {system: measure(system) for system in systems}
for system, result in scores.items():
    print("SYNTHETIC", system, result)

## Step 3 · Ask a judge one narrow question

Naming a downside needs interpretation. The judge sees the chosen option and its explanation. Its practice verdicts deliberately include one mistake: calling a benefit a cost. A judge from a competing model family can still be biased; inspect agreement with human ratings.

In [ ]:
RUBRIC = ("Does this answer name a cost or downside of the option it chose? "
          'Reply with JSON only, no code fences: {"names_cost": true or false}')
judge_script = [False, False, True, True, False, True, True, True]

def judge(option, answer, example):
    reply = ask(f"{RUBRIC}\nOption: {option}\nAnswer: {answer}", example=json.dumps({"names_cost": example}))
    return json.loads(reply)["names_cost"] is True

judged = {key: judge(run[0], run[2], example) for (key, run), example in zip(runs.items(), judge_script)}
for key, verdict in judged.items():
    print(key, "names a cost:", verdict)

## Step 4 · Calibrate against human ratings

A four-answer sample only demonstrates calibration. At 75% agreement the practice judge passes despite one error. A real evaluation needs a representative calibration set, error analysis, and a separate rating set.

In [ ]:
human = {("Qwen", "Support search"): False, ("DeepSeek", "Support search"): True,
         ("DeepSeek", "Order details"): False, ("Gemini", "Order details"): True}
agree = sum(judged[key] == rating for key, rating in human.items())
trusted = agree / len(human) >= 0.75
print(f"Judge-human agreement: {agree}/{len(human)}. Use judge metric: {trusted}")
for key, rating in human.items():
    if judged[key] != rating:
        print("Disagreement:", key, runs[key][2], "judge:", judged[key], "human:", rating)

## Step 5 · Compare with the best single-model fixture

Select the strongest baseline by label agreement and then valid citations. An untrusted judge metric remains `None`. This prints a comparison of fixtures, not a claim that any named model or council won an experiment.

In [ ]:
for system in systems:
    scores[system]["names_cost"] = sum(v for (s, _), v in judged.items() if s == system) if trusted else None
best = max(systems[:3], key=lambda s: (scores[s]["correct"], scores[s]["citation_valid"]))
for metric in ("correct", "citation_valid", "names_cost", "calls", "seconds"):
    print(f"{metric:<15} council {scores['Council'][metric]!s:<6} {best} fixture {scores[best][metric]}")
wins = [m for m in ("correct", "citation_valid", "names_cost")
        if scores["Council"][m] is not None and scores["Council"][m] > scores[best][m]]
ratio = scores["Council"]["calls"] / scores[best]["calls"]
print("Fixture comparison: council wins on", wins, "with", ratio, "times the illustrative calls.")

## What you should see

The DeepSeek and council fixtures each match 2/2 labels; Qwen and Gemini match 1/2. The practice judge agrees with the human on 3/4 samples. Its error makes the council and DeepSeek tie on stated costs. The council fixture has more valid citations and five times the illustrative calls. None of these numbers are empirical model results.

## Try it

1. Require 80% judge-human agreement. Why is `names_cost` now reported as `None`?
2. Give a false explanation a known citation ID. What extra check is needed beyond `citation_valid`?
3. Add slide 30B (a saga for the trip-booking flow) and 30D (event sourcing for historical balances), with explicit human labels and reasons.
4. Collect one real answer from each system on each scenario. Replace synthetic timings with measured wall time and report token usage separately from elapsed time.

## Takeaway

Separate fixture-based plumbing tests from empirical evaluation. Compare identical scenarios, calibrate the judge, and report trade-offs and cost. The microservices revision matters too: CQRS, API composition, sagas, and event sourcing solve different problems.

## Optional extension · Save the synthetic comparison with MLflow

After the core steps, run `%pip install -q mlflow==2.20.0` in a new cell, then this extra cell. This logs the **synthetic fixture** metrics, not new provider benchmarks. Missing/untrusted judge metrics are omitted, never converted to zero. Tracking is optional and is not part of the five core steps.

```python
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("synthetic-microservice-pattern-demo")
for system, metrics in scores.items():
    with mlflow.start_run(run_name=system):
        mlflow.log_params({"system": system, "data_kind": "synthetic teaching fixture"})
        mlflow.log_metrics({k: v for k, v in metrics.items() if v is not None})
```

In Colab, the tracking database `mlflow.db` lives in the runtime filesystem; download it before ending the session. On your own computer, use `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000` and open localhost:5000 there. A Colab runtime's localhost is not your laptop's localhost.